# Task 1: MLP forward/backward from scratch (NumPy only)

In [1]:
import numpy as np
from sklearn.datasets import make_classification

np.random.seed(0)


In [2]:
def init_params(sizes):
    params = {}
    for i in range(1, len(sizes)):
        params[f"W{i}"] = np.random.randn(sizes[i-1], sizes[i]) * np.sqrt(2/sizes[i-1])
        params[f"b{i}"] = np.zeros((1, sizes[i]))
    return params

def relu(x):
    return np.maximum(0, x)

def relu_grad(x):
    return (x > 0).astype(float)

def softmax(x):
    e = np.exp(x - np.max(x, axis=1, keepdims=True))
    return e / np.sum(e, axis=1, keepdims=True)


In [3]:
def forward(X, params, L):
    cache = {"A0": X}
    A = X
    for i in range(1, L):
        Z = A @ params[f"W{i}"] + params[f"b{i}"]
        A = relu(Z)
        cache[f"Z{i}"] = Z
        cache[f"A{i}"] = A
    ZL = A @ params[f"W{L}"] + params[f"b{L}"]
    AL = softmax(ZL)
    cache[f"Z{L}"] = ZL
    cache[f"A{L}"] = AL
    return AL, cache

def cross_entropy(Y_hat, Y):
    m = Y.shape[0]
    return -np.sum(Y * np.log(Y_hat + 1e-9)) / m


In [4]:
def backward(Y, params, cache, L):
    grads = {}
    m = Y.shape[0]
    dZ = cache[f"A{L}"] - Y
    for i in reversed(range(1, L+1)):
        A_prev = cache[f"A{i-1}"]
        grads[f"W{i}"] = A_prev.T @ dZ / m
        grads[f"b{i}"] = np.sum(dZ, axis=0, keepdims=True) / m
        if i > 1:
            dA_prev = dZ @ params[f"W{i}"].T
            dZ = dA_prev * relu_grad(cache[f"Z{i-1}"])
    return grads

def update(params, grads, lr, L):
    for i in range(1, L+1):
        params[f"W{i}"] -= lr * grads[f"W{i}"]
        params[f"b{i}"] -= lr * grads[f"b{i}"]
    return params


In [5]:
X, y = make_classification(n_samples=500, n_features=10, n_classes=3, n_informative=6, random_state=0)
Y = np.eye(3)[y]

sizes = [10, 16, 8, 3]
L = len(sizes) - 1
params = init_params(sizes)

for epoch in range(200):
    AL, cache = forward(X, params, L)
    loss = cross_entropy(AL, Y)
    grads = backward(Y, params, cache, L)
    params = update(params, grads, 0.1, L)
    if epoch % 50 == 0:
        print(epoch, loss)

print("final loss", loss)


0 1.5736742876269814
50 0.7552428205459832
100 0.6749379231552367
150 0.6190337011582496
final loss 0.5670866072992323
